In [3]:
import pandas as pd
import numpy as np
import os

RAW_FILE = "../data/raw/PS_20174392719_1491204439457_log.csv"

PROCESSED_DIR = "../data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Raw file:", RAW_FILE)
print("Processed directory:", PROCESSED_DIR)

Raw file: ../data/raw/PS_20174392719_1491204439457_log.csv
Processed directory: ../data/processed


In [4]:
step_data = pd.read_csv(
    RAW_FILE,
    usecols=["step"]
)

print("Minimum step:", step_data["step"].min())
print("Maximum step:", step_data["step"].max())
print("Unique steps:", step_data["step"].nunique())

Minimum step: 1
Maximum step: 743
Unique steps: 743


In [5]:
print(
    "Dataset sorted by step:",
    step_data["step"].is_monotonic_increasing
)

Dataset sorted by step: True


In [6]:
missing_values = {}

for chunk in pd.read_csv(RAW_FILE, chunksize=100_000):
    for column in chunk.columns:
        missing_values[column] = (
            missing_values.get(column, 0)
            + chunk[column].isna().sum()
        )

missing_values = pd.Series(missing_values)

print(missing_values)

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [7]:
duplicate_count = 0

for chunk in pd.read_csv(RAW_FILE, chunksize=100_000):
    duplicate_count += chunk.duplicated().sum()

print("Duplicate rows found within chunks:", duplicate_count)

Duplicate rows found within chunks: 0


In [8]:
dtype_check = pd.read_csv(
    RAW_FILE,
    nrows=10000
)

print(dtype_check.dtypes)

step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object


In [9]:
import pandas as pd
import numpy as np

RAW_FILE = "../data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(
    RAW_FILE,
    nrows=10000
)

print(df.shape)

(10000, 11)


In [10]:
df = df.sort_values(
    ["step"]
).reset_index(drop=True)

print(df[["step", "nameOrig", "amount"]].head(10))

   step     nameOrig    amount
0     1  C1231006815   9839.64
1     1  C1666544295   1864.28
2     1  C1305486145    181.00
3     1   C840083671    181.00
4     1  C2048537720  11668.14
5     1    C90045638   7817.71
6     1   C154988899   7107.77
7     1  C1912850431   7861.64
8     1  C1265012928   4024.36
9     1   C712410124   5337.77


In [11]:
df["user_transaction_count_before"] = (
    df.groupby("nameOrig").cumcount()
)

In [12]:
df[
    [
        "step",
        "nameOrig",
        "amount",
        "user_transaction_count_before"
    ]
].head(20)

,step,nameOrig,amount,user_transaction_count_before
0,1,C1231006815,9839.64,0
1,1,C1666544295,1864.28,0
2,1,C1305486145,181.00,0
3,1,C840083671,181.00,0
4,1,C2048537720,11668.14,0
5,1,C90045638,7817.71,0
6,1,C154988899,7107.77,0
7,1,C1912850431,7861.64,0
8,1,C1265012928,4024.36,0
9,1,C712410124,5337.77,0


In [13]:
df[
    [
        "nameOrig",
        "amount",
        "previous_transaction_amount"
    ]
].head(20)

KeyError: "['previous_transaction_amount'] not in index"

In [14]:
test_users = pd.DataFrame({
    "step": [1, 2, 3, 4, 5, 6],
    "nameOrig": ["C100", "C200", "C100", "C100", "C200", "C100"],
    "amount": [100, 500, 200, 300, 600, 1000]
})

test_users

,step,nameOrig,amount
0,1,C100,100
1,2,C200,500
2,3,C100,200
3,4,C100,300
4,5,C200,600
5,6,C100,1000


In [15]:
test_users["user_transaction_count_before"] = (
    test_users.groupby("nameOrig").cumcount()
)

test_users

,step,nameOrig,amount,user_transaction_count_before
0,1,C100,100,0
1,2,C200,500,0
2,3,C100,200,1
3,4,C100,300,2
4,5,C200,600,1
5,6,C100,1000,3


In [16]:
test_users["previous_transaction_amount"] = (
    test_users.groupby("nameOrig")["amount"].shift(1)
)

In [17]:
test_users[
    [
        "step",
        "nameOrig",
        "amount",
        "previous_transaction_amount"
    ]
]

,step,nameOrig,amount,previous_transaction_amount
0,1,C100,100,NaN
1,2,C200,500,NaN
2,3,C100,200,100.0
3,4,C100,300,200.0
4,5,C200,600,500.0
5,6,C100,1000,300.0


In [8]:
import pandas as pd
import numpy as np
RAW_FILE = "../data/raw/PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(
    RAW_FILE,
    nrows=10000
)

print(df.shape)


(10000, 11)


In [9]:
df["previous_transaction_amount"] = (
    df.groupby("nameOrig")["amount"].shift(1)
)

In [10]:
df[df["previous_transaction_amount"].notna()][
    [
        "step",
        "nameOrig",
        "amount",
        "previous_transaction_amount"
    ]
].head(20)

,step,nameOrig,amount,previous_transaction_amount


In [12]:
df["time_since_previous_transaction"] = (
    df.groupby("nameOrig")["step"].diff()
)

In [13]:
df[
    [
        "step",
        "nameOrig",
        "amount",
        "time_since_previous_transaction"
    ]
].head(20)

,step,nameOrig,amount,time_since_previous_transaction
0,1,C1231006815,9839.64,NaN
1,1,C1666544295,1864.28,NaN
2,1,C1305486145,181.00,NaN
3,1,C840083671,181.00,NaN
4,1,C2048537720,11668.14,NaN
5,1,C90045638,7817.71,NaN
6,1,C154988899,7107.77,NaN
7,1,C1912850431,7861.64,NaN
8,1,C1265012928,4024.36,NaN
9,1,C712410124,5337.77,NaN


In [15]:
import pandas as pd
import numpy as np

RAW_FILE = "../data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(
    RAW_FILE,
    nrows=10000
)

print(df.shape)

(10000, 11)


In [16]:
df["user_transaction_count_before"] = (
    df.groupby("nameOrig").cumcount()
)

In [17]:
df["previous_transaction_amount"] = (
    df.groupby("nameOrig")["amount"].shift(1)
)

In [18]:
df["time_since_previous_transaction"] = (
    df.groupby("nameOrig")["step"].diff()
)

In [19]:
df[
    [
        "step",
        "nameOrig",
        "amount",
        "user_transaction_count_before",
        "previous_transaction_amount",
        "time_since_previous_transaction"
    ]
].head(20)

,step,nameOrig,amount,user_transaction_count_before,previous_transaction_amount,time_since_previous_transaction
0,1,C1231006815,9839.64,0,NaN,NaN
1,1,C1666544295,1864.28,0,NaN,NaN
2,1,C1305486145,181.00,0,NaN,NaN
3,1,C840083671,181.00,0,NaN,NaN
4,1,C2048537720,11668.14,0,NaN,NaN
5,1,C90045638,7817.71,0,NaN,NaN
6,1,C154988899,7107.77,0,NaN,NaN
7,1,C1912850431,7861.64,0,NaN,NaN
8,1,C1265012928,4024.36,0,NaN,NaN
9,1,C712410124,5337.77,0,NaN,NaN


In [20]:
df["previous_amount_sum"] = (
    df.groupby("nameOrig")["amount"].cumsum()
    - df["amount"]
)

In [21]:
df["previous_amount_count"] = (
    df.groupby("nameOrig").cumcount()
)

In [22]:
df["previous_average_amount"] = (
    df["previous_amount_sum"]
    / df["previous_amount_count"].replace(0, np.nan)
)

In [23]:
df[
    [
        "step",
        "nameOrig",
        "amount",
        "previous_amount_count",
        "previous_average_amount"
    ]
].head(20)

,step,nameOrig,amount,previous_amount_count,previous_average_amount
0,1,C1231006815,9839.64,0,NaN
1,1,C1666544295,1864.28,0,NaN
2,1,C1305486145,181.00,0,NaN
3,1,C840083671,181.00,0,NaN
4,1,C2048537720,11668.14,0,NaN
5,1,C90045638,7817.71,0,NaN
6,1,C154988899,7107.77,0,NaN
7,1,C1912850431,7861.64,0,NaN
8,1,C1265012928,4024.36,0,NaN
9,1,C712410124,5337.77,0,NaN


In [24]:
df["amount_deviation"] = (
    df["amount"] - df["previous_average_amount"]
)

In [25]:
df["amount_to_previous_average"] = (
    df["amount"]
    / df["previous_average_amount"].replace(0, np.nan)
)

In [26]:
df[
    [
        "step",
        "nameOrig",
        "amount",
        "user_transaction_count_before",
        "previous_transaction_amount",
        "time_since_previous_transaction",
        "previous_average_amount",
        "amount_deviation",
        "amount_to_previous_average"
    ]
].head(20)

,step,nameOrig,amount,user_transaction_count_before,previous_transaction_amount,time_since_previous_transaction,previous_average_amount,amount_deviation,amount_to_previous_average
0,1,C1231006815,9839.64,0,NaN,NaN,NaN,NaN,NaN
1,1,C1666544295,1864.28,0,NaN,NaN,NaN,NaN,NaN
2,1,C1305486145,181.00,0,NaN,NaN,NaN,NaN,NaN
3,1,C840083671,181.00,0,NaN,NaN,NaN,NaN,NaN
4,1,C2048537720,11668.14,0,NaN,NaN,NaN,NaN,NaN
5,1,C90045638,7817.71,0,NaN,NaN,NaN,NaN,NaN
6,1,C154988899,7107.77,0,NaN,NaN,NaN,NaN,NaN
7,1,C1912850431,7861.64,0,NaN,NaN,NaN,NaN,NaN
8,1,C1265012928,4024.36,0,NaN,NaN,NaN,NaN,NaN
9,1,C712410124,5337.77,0,NaN,NaN,NaN,NaN,NaN


In [28]:
user_counts = df["nameOrig"].value_counts()

print("Users with more than 1 transaction:")
print(user_counts[user_counts > 1].head(20))

Users with more than 1 transaction:
Series([], Name: count, dtype: int64)


In [29]:
unique_users = set()

for chunk in pd.read_csv(
    RAW_FILE,
    usecols=["nameOrig"],
    chunksize=100_000
):
    unique_users.update(chunk["nameOrig"].unique())

print("Unique users:", len(unique_users))

Unique users: 6353307


In [30]:
print("Total transactions:", 6_362_620)
print("Repeated-user transactions possible:", 
      6_362_620 - len(unique_users))

Total transactions: 6362620
Repeated-user transactions possible: 9313


In [31]:
user_history = {}

print("User history dictionary created.")


User history dictionary created.


In [32]:
import os
import pandas as pd
import numpy as np

RAW_FILE = "../data/raw/PS_20174392719_1491204439457_log.csv"
PROCESSED_DIR = "../data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "feature_engineered.csv"
)

print("Output:", OUTPUT_FILE)


Output: ../data/processed\feature_engineered.csv


In [33]:
user_history = {}

print("User history initialized.")

User history initialized.


In [34]:
test_data = pd.DataFrame({
    "step": [1, 2, 3, 4, 5, 6],
    "nameOrig": ["C100", "C200", "C100", "C100", "C200", "C100"],
    "amount": [100, 500, 200, 300, 600, 1000]
})

test_data

,step,nameOrig,amount
0,1,C100,100
1,2,C200,500
2,3,C100,200
3,4,C100,300
4,5,C200,600
5,6,C100,1000


In [35]:
history = {}

counts = []
previous_amounts = []
previous_steps = []

for _, row in test_data.iterrows():

    user = row["nameOrig"]

    if user in history:
        counts.append(history[user]["count"])
        previous_amounts.append(history[user]["previous_amount"])
        previous_steps.append(history[user]["previous_step"])
    else:
        counts.append(0)
        previous_amounts.append(np.nan)
        previous_steps.append(np.nan)

    # Update history AFTER creating the current transaction's features
    if user not in history:
        history[user] = {
            "count": 1,
            "previous_amount": row["amount"],
            "previous_step": row["step"]
        }
    else:
        history[user]["count"] += 1
        history[user]["previous_amount"] = row["amount"]
        history[user]["previous_step"] = row["step"]

test_data["user_transaction_count_before"] = counts
test_data["previous_transaction_amount"] = previous_amounts
test_data["previous_step"] = previous_steps

test_data

,step,nameOrig,amount,user_transaction_count_before,previous_transaction_amount,previous_step
0,1,C100,100,0,NaN,NaN
1,2,C200,500,0,NaN,NaN
2,3,C100,200,1,100.0,1.0
3,4,C100,300,2,200.0,3.0
4,5,C200,600,1,500.0,2.0
5,6,C100,1000,3,300.0,4.0


In [36]:
test_data["time_since_previous_transaction"] = (
    test_data["step"] - test_data["previous_step"]
)

In [37]:
test_data[
    [
        "step",
        "nameOrig",
        "amount",
        "user_transaction_count_before",
        "previous_transaction_amount",
        "time_since_previous_transaction"
    ]
]

,step,nameOrig,amount,user_transaction_count_before,previous_transaction_amount,time_since_previous_transaction
0,1,C100,100,0,NaN,NaN
1,2,C200,500,0,NaN,NaN
2,3,C100,200,1,100.0,2.0
3,4,C100,300,2,200.0,1.0
4,5,C200,600,1,500.0,3.0
5,6,C100,1000,3,300.0,2.0


In [38]:
# Reset the user history
user_history = {}

print("User history reset.")

User history reset.


In [39]:
chunksize = 100_000

print(f"Processing chunks of {chunksize:,} rows")

Processing chunks of 100,000 rows


In [40]:
user_history = {}
chunksize = 100_000

print("Ready for full feature processing.")

Ready for full feature processing.


In [41]:
import pandas as pd
import numpy as np
import os

RAW_FILE = "../data/raw/PS_20174392719_1491204439457_log.csv"
PROCESSED_DIR = "../data/processed"
OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "feature_engineered.csv"
)

os.makedirs(PROCESSED_DIR, exist_ok=True)

# Remove old output if it exists
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

chunksize = 100_000

# Stores previous behaviour for each sender
user_history = {}

first_chunk = True
total_processed = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(RAW_FILE, chunksize=chunksize),
    start=1
):

    # --------------------------------------------------
    # 1. Previous transaction count
    # --------------------------------------------------

    user_counts = chunk["nameOrig"].map(
        lambda user: user_history.get(user, {}).get("count", 0)
    )

    chunk["user_transaction_count_before"] = user_counts

    # --------------------------------------------------
    # 2. Previous transaction amount
    # --------------------------------------------------

    chunk["previous_transaction_amount"] = chunk["nameOrig"].map(
        lambda user: user_history.get(user, {}).get(
            "previous_amount", np.nan
        )
    )

    # --------------------------------------------------
    # 3. Previous transaction step
    # --------------------------------------------------

    previous_step = chunk["nameOrig"].map(
        lambda user: user_history.get(user, {}).get(
            "previous_step", np.nan
        )
    )

    chunk["time_since_previous_transaction"] = (
        chunk["step"] - previous_step
    )

    # --------------------------------------------------
    # 4. Previous average transaction amount
    # --------------------------------------------------

    previous_sum = chunk["nameOrig"].map(
        lambda user: user_history.get(user, {}).get(
            "amount_sum", 0.0
        )
    )

    previous_count = chunk["user_transaction_count_before"]

    chunk["previous_average_amount"] = np.where(
        previous_count > 0,
        previous_sum / previous_count,
        np.nan
    )

    # --------------------------------------------------
    # 5. Amount deviation
    # --------------------------------------------------

    chunk["amount_deviation"] = (
        chunk["amount"]
        - chunk["previous_average_amount"]
    )

    # --------------------------------------------------
    # 6. Amount compared with historical average
    # --------------------------------------------------

    chunk["amount_to_previous_average"] = (
        chunk["amount"]
        / chunk["previous_average_amount"].replace(0, np.nan)
    )

    # --------------------------------------------------
    # 7. Balance depletion
    # --------------------------------------------------

    chunk["balance_depletion"] = (
        chunk["oldbalanceOrg"]
        - chunk["newbalanceOrig"]
    )

    # --------------------------------------------------
    # 8. Amount-to-balance ratio
    # --------------------------------------------------

    chunk["amount_to_balance_ratio"] = (
        chunk["amount"]
        / (chunk["oldbalanceOrg"] + 1)
    )

    # --------------------------------------------------
    # 9. Update user history
    # --------------------------------------------------

    for user, group in chunk.groupby("nameOrig", sort=False):

        last_row = group.iloc[-1]

        previous = user_history.get(
            user,
            {
                "count": 0,
                "amount_sum": 0.0,
                "previous_amount": np.nan,
                "previous_step": np.nan
            }
        )

        previous["count"] += len(group)

        previous["amount_sum"] += group["amount"].sum()

        previous["previous_amount"] = last_row["amount"]

        previous["previous_step"] = last_row["step"]

        user_history[user] = previous

    # --------------------------------------------------
    # 10. Save processed chunk
    # --------------------------------------------------

    chunk.to_csv(
        OUTPUT_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    total_processed += len(chunk)

    print(
        f"Chunk {chunk_number}/64 processed | "
        f"Rows: {total_processed:,}"
    )

print("\nFeature engineering completed!")
print(f"Total rows processed: {total_processed:,}")
print(f"Output file: {OUTPUT_FILE}")

Chunk 1/64 processed | Rows: 100,000
Chunk 2/64 processed | Rows: 200,000
Chunk 3/64 processed | Rows: 300,000
Chunk 4/64 processed | Rows: 400,000
Chunk 5/64 processed | Rows: 500,000
Chunk 6/64 processed | Rows: 600,000
Chunk 7/64 processed | Rows: 700,000
Chunk 8/64 processed | Rows: 800,000
Chunk 9/64 processed | Rows: 900,000
Chunk 10/64 processed | Rows: 1,000,000
Chunk 11/64 processed | Rows: 1,100,000
Chunk 12/64 processed | Rows: 1,200,000
Chunk 13/64 processed | Rows: 1,300,000
Chunk 14/64 processed | Rows: 1,400,000
Chunk 15/64 processed | Rows: 1,500,000
Chunk 16/64 processed | Rows: 1,600,000
Chunk 17/64 processed | Rows: 1,700,000
Chunk 18/64 processed | Rows: 1,800,000
Chunk 19/64 processed | Rows: 1,900,000
Chunk 20/64 processed | Rows: 2,000,000
Chunk 21/64 processed | Rows: 2,100,000
Chunk 22/64 processed | Rows: 2,200,000
Chunk 23/64 processed | Rows: 2,300,000
Chunk 24/64 processed | Rows: 2,400,000
Chunk 25/64 processed | Rows: 2,500,000
Chunk 26/64 processed | Row

In [42]:
import os

file_size = os.path.getsize(OUTPUT_FILE) / (1024 ** 3)

print(f"Feature-engineered file size: {file_size:.2f} GB")

Feature-engineered file size: 0.66 GB


In [43]:
check_df = pd.read_csv(
    OUTPUT_FILE,
    nrows=5
)

print(check_df.columns.tolist())

['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'user_transaction_count_before', 'previous_transaction_amount', 'time_since_previous_transaction', 'previous_average_amount', 'amount_deviation', 'amount_to_previous_average', 'balance_depletion', 'amount_to_balance_ratio']


In [44]:
processed_sample = pd.read_csv(
    OUTPUT_FILE,
    nrows=10000
)

print(processed_sample.shape)

(10000, 19)


In [45]:
processed_sample[
    [
        "user_transaction_count_before",
        "previous_transaction_amount",
        "time_since_previous_transaction",
        "previous_average_amount",
        "amount_deviation",
        "amount_to_previous_average",
        "balance_depletion",
        "amount_to_balance_ratio"
    ]
].isna().sum()

user_transaction_count_before          0
previous_transaction_amount        10000
time_since_previous_transaction    10000
previous_average_amount            10000
amount_deviation                   10000
amount_to_previous_average         10000
balance_depletion                      0
amount_to_balance_ratio                0
dtype: int64

In [46]:
history_check = pd.read_csv(
    OUTPUT_FILE,
    usecols=[
        "nameOrig",
        "user_transaction_count_before"
    ]
)

In [47]:
history_check[
    "user_transaction_count_before"
].value_counts().sort_index().head(20)

user_transaction_count_before
0    6353430
1       9175
2         15
Name: count, dtype: int64

In [48]:
history_check.sort_values(
    "user_transaction_count_before",
    ascending=False
).head(20)

,nameOrig,user_transaction_count_before
2727335,C1677795071,2
2181883,C400299098,2
1432799,C1999539787,2
2895229,C1530544995,2
4359348,C1784010646,2
4652048,C1976208114,2
4829222,C1065307291,2
5039138,C2098525306,2
5146021,C545315117,2
5469706,C2051359467,2


In [49]:
fraud_rows = []

for chunk in pd.read_csv(
    OUTPUT_FILE,
    chunksize=100_000
):
    fraud = chunk[chunk["isFraud"] == 1]

    if len(fraud) > 0:
        fraud_rows.append(fraud)

fraud_df = pd.concat(
    fraud_rows,
    ignore_index=True
)

print("Fraud transactions:", len(fraud_df))

Fraud transactions: 8213


In [50]:
fraud_df[
    [
        "step",
        "type",
        "amount",
        "user_transaction_count_before",
        "previous_transaction_amount",
        "time_since_previous_transaction",
        "previous_average_amount",
        "amount_deviation",
        "amount_to_previous_average",
        "balance_depletion",
        "amount_to_balance_ratio"
    ]
].head(20)

,step,type,amount,user_transaction_count_before,previous_transaction_amount,time_since_previous_transaction,previous_average_amount,amount_deviation,amount_to_previous_average,balance_depletion,amount_to_balance_ratio
0,1,TRANSFER,181.00,0,NaN,NaN,NaN,NaN,NaN,181.00,0.994505
1,1,CASH_OUT,181.00,0,NaN,NaN,NaN,NaN,NaN,181.00,0.994505
2,1,TRANSFER,2806.00,0,NaN,NaN,NaN,NaN,NaN,2806.00,0.999644
3,1,CASH_OUT,2806.00,0,NaN,NaN,NaN,NaN,NaN,2806.00,0.999644
4,1,TRANSFER,20128.00,0,NaN,NaN,NaN,NaN,NaN,20128.00,0.999950
5,1,CASH_OUT,20128.00,0,NaN,NaN,NaN,NaN,NaN,20128.00,0.999950
6,1,CASH_OUT,416001.33,0,NaN,NaN,NaN,NaN,NaN,0.00,416001.330000
7,1,TRANSFER,1277212.77,0,NaN,NaN,NaN,NaN,NaN,1277212.77,0.999999
8,1,CASH_OUT,1277212.77,0,NaN,NaN,NaN,NaN,NaN,1277212.77,0.999999
9,1,TRANSFER,35063.63,0,NaN,NaN,NaN,NaN,NaN,35063.63,0.999971


In [51]:
history_check = pd.read_csv(
    OUTPUT_FILE,
    usecols=[
        "nameOrig",
        "user_transaction_count_before"
    ]
)

print(
    history_check[
        "user_transaction_count_before"
    ].value_counts().sort_index().head(20)
)

user_transaction_count_before
0    6353430
1       9175
2         15
Name: count, dtype: int64


In [52]:
print(
    "Maximum previous transaction count:",
    history_check["user_transaction_count_before"].max()
)

Maximum previous transaction count: 2


In [53]:
repeated_users = history_check[
    history_check["user_transaction_count_before"] > 0
]["nameOrig"].unique()

print("Number of users with history:", len(repeated_users))
print("First few:", repeated_users[:10])

Number of users with history: 9175
First few: <ArrowStringArray>
['C1709295811',   'C44568807',  'C260230637',  'C745009740', 'C1842781381',
  'C199116739',  'C779875094',  'C189326840', 'C1250194175',  'C746558292']
Length: 10, dtype: str


In [54]:
example_user = repeated_users[0]

user_example = pd.read_csv(
    OUTPUT_FILE,
    usecols=[
        "step",
        "nameOrig",
        "amount",
        "user_transaction_count_before",
        "previous_transaction_amount",
        "time_since_previous_transaction",
        "previous_average_amount",
        "amount_deviation",
        "amount_to_previous_average"
    ]
)

user_example[
    user_example["nameOrig"] == example_user
].head(20)

,step,amount,nameOrig,user_transaction_count_before,previous_transaction_amount,time_since_previous_transaction,previous_average_amount,amount_deviation,amount_to_previous_average
89078,10,497725.93,C1709295811,0,NaN,NaN,NaN,NaN,NaN
115385,11,17670.78,C1709295811,1,497725.93,1.0,497725.93,-480055.15,0.035503


In [55]:
PROCESSED_FILE = "../data/processed/feature_engineered.csv"

TRAIN_FILE = "../data/processed/train_data.csv"
VALIDATION_FILE = "../data/processed/validation_data.csv"
TEST_FILE = "../data/processed/test_data.csv"

In [56]:
steps = pd.read_csv(
    PROCESSED_FILE,
    usecols=["step"]
)

unique_steps = sorted(steps["step"].unique())

total_steps = len(unique_steps)

train_end_index = int(total_steps * 0.70)
validation_end_index = int(total_steps * 0.85)

train_end_step = unique_steps[train_end_index - 1]
validation_end_step = unique_steps[validation_end_index - 1]

print("Total steps:", total_steps)
print("Training ends at step:", train_end_step)
print("Validation ends at step:", validation_end_step)
print("Test ends at step:", unique_steps[-1])

Total steps: 743
Training ends at step: 520
Validation ends at step: 631
Test ends at step: 743


In [57]:
import os

for file in [TRAIN_FILE, VALIDATION_FILE, TEST_FILE]:
    if os.path.exists(file):
        os.remove(file)

train_rows = 0
validation_rows = 0
test_rows = 0

first_train = True
first_validation = True
first_test = True

for chunk in pd.read_csv(
    PROCESSED_FILE,
    chunksize=100_000
):

    train_chunk = chunk[
        chunk["step"] <= train_end_step
    ]

    validation_chunk = chunk[
        (chunk["step"] > train_end_step) &
        (chunk["step"] <= validation_end_step)
    ]

    test_chunk = chunk[
        chunk["step"] > validation_end_step
    ]

    if len(train_chunk) > 0:
        train_chunk.to_csv(
            TRAIN_FILE,
            mode="w" if first_train else "a",
            header=first_train,
            index=False
        )
        first_train = False
        train_rows += len(train_chunk)

    if len(validation_chunk) > 0:
        validation_chunk.to_csv(
            VALIDATION_FILE,
            mode="w" if first_validation else "a",
            header=first_validation,
            index=False
        )
        first_validation = False
        validation_rows += len(validation_chunk)

    if len(test_chunk) > 0:
        test_chunk.to_csv(
            TEST_FILE,
            mode="w" if first_test else "a",
            header=first_test,
            index=False
        )
        first_test = False
        test_rows += len(test_chunk)

print("Train rows:", train_rows)
print("Validation rows:", validation_rows)
print("Test rows:", test_rows)
print("Total:", train_rows + validation_rows + test_rows)

Train rows: 6082007
Validation rows: 191147
Test rows: 89466
Total: 6362620


In [58]:
train_check = pd.read_csv(
    TRAIN_FILE,
    usecols=["step", "isFraud"]
)

validation_check = pd.read_csv(
    VALIDATION_FILE,
    usecols=["step", "isFraud"]
)

test_check = pd.read_csv(
    TEST_FILE,
    usecols=["step", "isFraud"]
)

print(
    "Train:",
    train_check["step"].min(),
    "→",
    train_check["step"].max()
)

print(
    "Validation:",
    validation_check["step"].min(),
    "→",
    validation_check["step"].max()
)

print(
    "Test:",
    test_check["step"].min(),
    "→",
    test_check["step"].max()
)

Train: 1 → 520
Validation: 521 → 631
Test: 632 → 743


In [59]:
print("TRAIN")
print(train_check["isFraud"].value_counts())
print()

print("VALIDATION")
print(validation_check["isFraud"].value_counts())
print()

print("TEST")
print(test_check["isFraud"].value_counts())

TRAIN
isFraud
0    6076226
1       5781
Name: count, dtype: int64

VALIDATION
isFraud
0    189967
1      1180
Name: count, dtype: int64

TEST
isFraud
0    88214
1     1252
Name: count, dtype: int64


In [60]:
print("Train fraud rate:",
      train_check["isFraud"].mean() * 100)

print("Validation fraud rate:",
      validation_check["isFraud"].mean() * 100)

print("Test fraud rate:",
      test_check["isFraud"].mean() * 100)

Train fraud rate: 0.09505086067806236
Validation fraud rate: 0.6173259323975788
Test fraud rate: 1.3994143026401091


In [61]:
test_receiver = pd.DataFrame({
    "step": [1, 2, 3, 4, 5, 6],
    "nameOrig": ["C100", "C200", "C300", "C400", "C500", "C600"],
    "nameDest": ["M100", "M200", "M100", "M100", "M200", "M100"],
    "amount": [100, 500, 200, 300, 600, 1000]
})

test_receiver

,step,nameOrig,nameDest,amount
0,1,C100,M100,100
1,2,C200,M200,500
2,3,C300,M100,200
3,4,C400,M100,300
4,5,C500,M200,600
5,6,C600,M100,1000


In [62]:
test_receiver["receiver_transaction_count_before"] = (
    test_receiver.groupby("nameDest").cumcount()
)

In [63]:
test_receiver[
    [
        "step",
        "nameDest",
        "amount",
        "receiver_transaction_count_before"
    ]
]

,step,nameDest,amount,receiver_transaction_count_before
0,1,M100,100,0
1,2,M200,500,0
2,3,M100,200,1
3,4,M100,300,2
4,5,M200,600,1
5,6,M100,1000,3


In [64]:
test_receiver["receiver_transaction_count_before"] = (
    test_receiver.groupby("nameDest").cumcount()
)

In [65]:
test_receiver[
    [
        "step",
        "nameDest",
        "amount",
        "receiver_transaction_count_before"
    ]
]

,step,nameDest,amount,receiver_transaction_count_before
0,1,M100,100,0
1,2,M200,500,0
2,3,M100,200,1
3,4,M100,300,2
4,5,M200,600,1
5,6,M100,1000,3


In [66]:
test_receiver["receiver_previous_amount"] = (
    test_receiver.groupby("nameDest")["amount"].shift(1)
)



In [67]:
test_receiver[
    [
        "step",
        "nameDest",
        "amount",
        "receiver_transaction_count_before",
        "receiver_previous_amount"
    ]
]

,step,nameDest,amount,receiver_transaction_count_before,receiver_previous_amount
0,1,M100,100,0,NaN
1,2,M200,500,0,NaN
2,3,M100,200,1,100.0
3,4,M100,300,2,200.0
4,5,M200,600,1,500.0
5,6,M100,1000,3,300.0


In [68]:
test_receiver["receiver_first_step"] = (
    test_receiver.groupby("nameDest")["step"].transform("first")
)

In [69]:
elapsed_steps = (
    test_receiver["step"]
    - test_receiver["receiver_first_step"]
)

test_receiver["receiver_transaction_frequency"] = (
    test_receiver["receiver_transaction_count_before"]
    / elapsed_steps.replace(0, np.nan)
)

In [70]:
test_receiver[
    [
        "step",
        "nameDest",
        "amount",
        "receiver_transaction_count_before",
        "receiver_previous_amount",
        "receiver_transaction_frequency"
    ]
]

,step,nameDest,amount,receiver_transaction_count_before,receiver_previous_amount,receiver_transaction_frequency
0,1,M100,100,0,NaN,NaN
1,2,M200,500,0,NaN,NaN
2,3,M100,200,1,100.0,0.500000
3,4,M100,300,2,200.0,0.666667
4,5,M200,600,1,500.0,0.333333
5,6,M100,1000,3,300.0,0.600000


In [71]:
test_type = pd.DataFrame({
    "step": [1, 2, 3, 4, 5, 6],
    "nameOrig": [
        "C100",
        "C100",
        "C100",
        "C200",
        "C200",
        "C100"
    ],
    "type": [
        "TRANSFER",
        "CASH_OUT",
        "TRANSFER",
        "CASH_OUT",
        "TRANSFER",
        "CASH_OUT"
    ],
    "amount": [
        100,
        200,
        300,
        400,
        500,
        600
    ]
})

test_type

,step,nameOrig,type,amount
0,1,C100,TRANSFER,100
1,2,C100,CASH_OUT,200
2,3,C100,TRANSFER,300
3,4,C200,CASH_OUT,400
4,5,C200,TRANSFER,500
5,6,C100,CASH_OUT,600


In [72]:
test_type["user_transfer_count_before"] = (
    test_type["type"].eq("TRANSFER")
    .groupby(test_type["nameOrig"])
    .cumsum()
    .shift(1)
    .fillna(0)
)

In [73]:
test_type["user_cashout_count_before"] = (
    test_type["type"].eq("CASH_OUT")
    .groupby(test_type["nameOrig"])
    .cumsum()
    .shift(1)
    .fillna(0)
)

In [74]:
test_type[
    [
        "step",
        "nameOrig",
        "type",
        "user_transfer_count_before",
        "user_cashout_count_before"
    ]
]

,step,nameOrig,type,user_transfer_count_before,user_cashout_count_before
0,1,C100,TRANSFER,0.0,0.0
1,2,C100,CASH_OUT,1.0,0.0
2,3,C100,TRANSFER,1.0,1.0
3,4,C200,CASH_OUT,2.0,1.0
4,5,C200,TRANSFER,0.0,1.0
5,6,C100,CASH_OUT,1.0,1.0


In [75]:
test_type = test_type.drop(
    columns=[
        "user_transfer_count_before",
        "user_cashout_count_before"
    ]
)

In [76]:
test_type["user_transfer_count_before"] = (
    test_type["type"]
    .eq("TRANSFER")
    .astype(int)
    .groupby(test_type["nameOrig"])
    .transform(lambda x: x.cumsum().shift(1).fillna(0))
)

In [77]:
test_type["user_cashout_count_before"] = (
    test_type["type"]
    .eq("CASH_OUT")
    .astype(int)
    .groupby(test_type["nameOrig"])
    .transform(lambda x: x.cumsum().shift(1).fillna(0))
)

In [78]:
test_type[
    [
        "step",
        "nameOrig",
        "type",
        "user_transfer_count_before",
        "user_cashout_count_before"
    ]
]

,step,nameOrig,type,user_transfer_count_before,user_cashout_count_before
0,1,C100,TRANSFER,0.0,0.0
1,2,C100,CASH_OUT,1.0,0.0
2,3,C100,TRANSFER,1.0,1.0
3,4,C200,CASH_OUT,0.0,0.0
4,5,C200,TRANSFER,0.0,1.0
5,6,C100,CASH_OUT,2.0,1.0


In [79]:
test_velocity = pd.DataFrame({
    "step": [1, 1, 1, 2, 2, 3, 3],
    "nameOrig": [
        "C100",
        "C200",
        "C300",
        "C400",
        "C500",
        "C600",
        "C700"
    ],
    "amount": [
        100,
        200,
        300,
        400,
        500,
        600,
        700
    ]
})

test_velocity

,step,nameOrig,amount
0,1,C100,100
1,1,C200,200
2,1,C300,300
3,2,C400,400
4,2,C500,500
5,3,C600,600
6,3,C700,700


In [80]:
test_velocity["transaction_velocity"] = (
    test_velocity.groupby("step").cumcount()
)

In [81]:
test_velocity[
    [
        "step",
        "nameOrig",
        "amount",
        "transaction_velocity"
    ]
]

,step,nameOrig,amount,transaction_velocity
0,1,C100,100,0
1,1,C200,200,1
2,1,C300,300,2
3,2,C400,400,0
4,2,C500,500,1
5,3,C600,600,0
6,3,C700,700,1


In [82]:
import os

RAW_FILE = "../data/raw/PS_20174392719_1491204439457_log.csv"
OUTPUT_FILE = "../data/processed/feature_engineered.csv"

if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

print("Old feature-engineered file removed.")
print("Ready to regenerate.")

Old feature-engineered file removed.
Ready to regenerate.


In [83]:
user_history = {}
receiver_history = {}

first_chunk = True
total_processed = 0

print("Sender and receiver histories reset.")

Sender and receiver histories reset.


In [84]:
import pandas as pd
import numpy as np
import os

RAW_FILE = "../data/raw/PS_20174392719_1491204439457_log.csv"
OUTPUT_FILE = "../data/processed/feature_engineered.csv"

chunksize = 100_000

first_chunk = True
total_processed = 0

# Sender history
user_history = {}

# Receiver history
receiver_history = {}

for chunk_number, chunk in enumerate(
    pd.read_csv(RAW_FILE, chunksize=chunksize),
    start=1
):

    # ==================================================
    # 1. SENDER BEHAVIOUR
    # ==================================================

    chunk["user_transaction_count_before"] = (
        chunk["nameOrig"].map(
            lambda user: user_history.get(user, {}).get("count", 0)
        )
    )

    chunk["previous_transaction_amount"] = (
        chunk["nameOrig"].map(
            lambda user: user_history.get(user, {}).get(
                "previous_amount", np.nan
            )
        )
    )

    previous_step = chunk["nameOrig"].map(
        lambda user: user_history.get(user, {}).get(
            "previous_step", np.nan
        )
    )

    chunk["time_since_previous_transaction"] = (
        chunk["step"] - previous_step
    )

    previous_sum = chunk["nameOrig"].map(
        lambda user: user_history.get(user, {}).get(
            "amount_sum", 0.0
        )
    )

    previous_count = chunk["user_transaction_count_before"]

    chunk["previous_average_amount"] = np.where(
        previous_count > 0,
        previous_sum / previous_count,
        np.nan
    )

    chunk["amount_deviation"] = (
        chunk["amount"]
        - chunk["previous_average_amount"]
    )

    chunk["amount_to_previous_average"] = (
        chunk["amount"]
        / chunk["previous_average_amount"].replace(0, np.nan)
    )

    # ==================================================
    # 2. BALANCE BEHAVIOUR
    # ==================================================

    chunk["balance_depletion"] = (
        chunk["oldbalanceOrg"]
        - chunk["newbalanceOrig"]
    )

    chunk["amount_to_balance_ratio"] = (
        chunk["amount"]
        / (chunk["oldbalanceOrg"] + 1)
    )

    # ==================================================
    # 3. RECEIVER BEHAVIOUR
    # ==================================================

    chunk["receiver_transaction_count_before"] = (
        chunk["nameDest"].map(
            lambda receiver: receiver_history.get(
                receiver, {}
            ).get("count", 0)
        )
    )

    chunk["receiver_previous_amount"] = (
        chunk["nameDest"].map(
            lambda receiver: receiver_history.get(
                receiver, {}
            ).get("previous_amount", np.nan)
        )
    )

    receiver_first_step = chunk["nameDest"].map(
        lambda receiver: receiver_history.get(
            receiver, {}
        ).get("first_step", np.nan)
    )

    elapsed_receiver_steps = (
        chunk["step"] - receiver_first_step
    )

    chunk["receiver_transaction_frequency"] = (
        chunk["receiver_transaction_count_before"]
        / elapsed_receiver_steps.replace(0, np.nan)
    )

    # ==================================================
    # 4. USER TRANSACTION TYPE BEHAVIOUR
    # ==================================================

    chunk["user_transfer_count_before"] = (
        chunk["nameOrig"].map(
            lambda user: user_history.get(user, {}).get(
                "transfer_count", 0
            )
        )
    )

    chunk["user_cashout_count_before"] = (
        chunk["nameOrig"].map(
            lambda user: user_history.get(user, {}).get(
                "cashout_count", 0
            )
        )
    )

    # ==================================================
    # 5. TRANSACTION VELOCITY
    # ==================================================

    # Number of transactions already seen at this step
    chunk["transaction_velocity"] = (
        chunk.groupby("step").cumcount()
    )

    # ==================================================
    # 6. UPDATE SENDER HISTORY
    # ==================================================

    for user, group in chunk.groupby(
        "nameOrig",
        sort=False
    ):

        last_row = group.iloc[-1]

        previous = user_history.get(
            user,
            {
                "count": 0,
                "amount_sum": 0.0,
                "previous_amount": np.nan,
                "previous_step": np.nan,
                "transfer_count": 0,
                "cashout_count": 0
            }
        )

        previous["count"] += len(group)

        previous["amount_sum"] += group["amount"].sum()

        previous["previous_amount"] = last_row["amount"]

        previous["previous_step"] = last_row["step"]

        previous["transfer_count"] += (
            (group["type"] == "TRANSFER").sum()
        )

        previous["cashout_count"] += (
            (group["type"] == "CASH_OUT").sum()
        )

        user_history[user] = previous

    # ==================================================
    # 7. UPDATE RECEIVER HISTORY
    # ==================================================

    for receiver, group in chunk.groupby(
        "nameDest",
        sort=False
    ):

        last_row = group.iloc[-1]

        if receiver not in receiver_history:

            receiver_history[receiver] = {
                "count": 0,
                "previous_amount": np.nan,
                "first_step": last_row["step"]
            }

        receiver_history[receiver]["count"] += len(group)

        receiver_history[receiver]["previous_amount"] = (
            last_row["amount"]
        )

    # ==================================================
    # 8. SAVE CHUNK
    # ==================================================

    chunk.to_csv(
        OUTPUT_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    total_processed += len(chunk)

    print(
        f"Chunk {chunk_number}/64 processed | "
        f"Rows: {total_processed:,}"
    )

print("\n====================================")
print("Feature engineering completed!")
print("Total rows:", total_processed)
print("Output:", OUTPUT_FILE)
print("====================================")

Chunk 1/64 processed | Rows: 100,000
Chunk 2/64 processed | Rows: 200,000
Chunk 3/64 processed | Rows: 300,000
Chunk 4/64 processed | Rows: 400,000
Chunk 5/64 processed | Rows: 500,000
Chunk 6/64 processed | Rows: 600,000
Chunk 7/64 processed | Rows: 700,000
Chunk 8/64 processed | Rows: 800,000
Chunk 9/64 processed | Rows: 900,000
Chunk 10/64 processed | Rows: 1,000,000
Chunk 11/64 processed | Rows: 1,100,000
Chunk 12/64 processed | Rows: 1,200,000
Chunk 13/64 processed | Rows: 1,300,000
Chunk 14/64 processed | Rows: 1,400,000
Chunk 15/64 processed | Rows: 1,500,000
Chunk 16/64 processed | Rows: 1,600,000
Chunk 17/64 processed | Rows: 1,700,000
Chunk 18/64 processed | Rows: 1,800,000
Chunk 19/64 processed | Rows: 1,900,000
Chunk 20/64 processed | Rows: 2,000,000
Chunk 21/64 processed | Rows: 2,100,000
Chunk 22/64 processed | Rows: 2,200,000
Chunk 23/64 processed | Rows: 2,300,000
Chunk 24/64 processed | Rows: 2,400,000
Chunk 25/64 processed | Rows: 2,500,000
Chunk 26/64 processed | Row

In [85]:
check_df = pd.read_csv(
    "../data/processed/feature_engineered.csv",
    nrows=1000
)

print("Rows:", check_df.shape[0])
print("Columns:", check_df.shape[1])
print(check_df.columns.tolist())

Rows: 1000
Columns: 25
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'user_transaction_count_before', 'previous_transaction_amount', 'time_since_previous_transaction', 'previous_average_amount', 'amount_deviation', 'amount_to_previous_average', 'balance_depletion', 'amount_to_balance_ratio', 'receiver_transaction_count_before', 'receiver_previous_amount', 'receiver_transaction_frequency', 'user_transfer_count_before', 'user_cashout_count_before', 'transaction_velocity']


In [86]:
new_features = [
    "receiver_transaction_count_before",
    "receiver_previous_amount",
    "receiver_transaction_frequency",
    "user_transfer_count_before",
    "user_cashout_count_before",
    "transaction_velocity"
]

print(check_df[new_features].head(20))

    receiver_transaction_count_before  receiver_previous_amount  \
0                                   0                       NaN   
1                                   0                       NaN   
2                                   0                       NaN   
3                                   0                       NaN   
4                                   0                       NaN   
5                                   0                       NaN   
6                                   0                       NaN   
7                                   0                       NaN   
8                                   0                       NaN   
9                                   0                       NaN   
10                                  0                       NaN   
11                                  0                       NaN   
12                                  0                       NaN   
13                                  0                       Na

In [87]:
print(check_df[new_features].describe())

       receiver_transaction_count_before  receiver_previous_amount  \
count                             1000.0                       0.0   
mean                                 0.0                       NaN   
std                                  0.0                       NaN   
min                                  0.0                       NaN   
25%                                  0.0                       NaN   
50%                                  0.0                       NaN   
75%                                  0.0                       NaN   
max                                  0.0                       NaN   

       receiver_transaction_frequency  user_transfer_count_before  \
count                             0.0                      1000.0   
mean                              NaN                         0.0   
std                               NaN                         0.0   
min                               NaN                         0.0   
25%                     

In [88]:
print(check_df[new_features].isna().sum())

receiver_transaction_count_before       0
receiver_previous_amount             1000
receiver_transaction_frequency       1000
user_transfer_count_before              0
user_cashout_count_before               0
transaction_velocity                    0
dtype: int64


In [89]:
row_count = 0

for chunk in pd.read_csv(
    "../data/processed/feature_engineered.csv",
    chunksize=100_000
):
    row_count += len(chunk)

print("Total rows:", row_count)

Total rows: 6362620


In [90]:
new_features = [
    "receiver_transaction_count_before",
    "receiver_previous_amount",
    "receiver_transaction_frequency",
    "user_transfer_count_before",
    "user_cashout_count_before",
    "transaction_velocity"
]

feature_stats = {
    feature: {
        "non_zero": 0,
        "non_null": 0,
        "max": 0
    }
    for feature in new_features
}

for chunk in pd.read_csv(
    "../data/processed/feature_engineered.csv",
    usecols=new_features,
    chunksize=100_000
):
    
    for feature in new_features:
        
        feature_series = chunk[feature]
        
        feature_stats[feature]["non_zero"] += (
            (feature_series.fillna(0) > 0).sum()
        )
        
        feature_stats[feature]["non_null"] += (
            feature_series.notna().sum()
        )
        
        current_max = feature_series.max()
        
        if pd.notna(current_max):
            feature_stats[feature]["max"] = max(
                feature_stats[feature]["max"],
                current_max
            )

print(pd.DataFrame(feature_stats).T)

                                    non_zero   non_null          max
receiver_transaction_count_before  3502229.0  6362620.0       112.00
receiver_previous_amount           3502223.0  3502229.0  73823490.36
receiver_transaction_frequency     3492110.0  3492110.0        78.00
user_transfer_count_before             748.0  6362620.0         1.00
user_cashout_count_before             3293.0  6362620.0         2.00
transaction_velocity               6361814.0  6362620.0     51351.00


In [91]:
print(
    "Maximum receiver history:",
    feature_stats["receiver_transaction_count_before"]["max"]
)

print(
    "Maximum transfer history:",
    feature_stats["user_transfer_count_before"]["max"]
)

print(
    "Maximum cash-out history:",
    feature_stats["user_cashout_count_before"]["max"]
)

print(
    "Maximum transaction velocity:",
    feature_stats["transaction_velocity"]["max"]
)

Maximum receiver history: 112
Maximum transfer history: 1
Maximum cash-out history: 2
Maximum transaction velocity: 51351


In [92]:
import os
import pandas as pd

PROCESSED_FILE = "../data/processed/feature_engineered.csv"

TRAIN_FILE = "../data/processed/train_data.csv"
VALIDATION_FILE = "../data/processed/validation_data.csv"
TEST_FILE = "../data/processed/test_data.csv"

In [93]:
TRAIN_END_STEP = 520
VALIDATION_END_STEP = 631

print("Train: 1 →", TRAIN_END_STEP)
print("Validation:", TRAIN_END_STEP + 1, "→", VALIDATION_END_STEP)
print("Test:", VALIDATION_END_STEP + 1, "→ 743")

Train: 1 → 520
Validation: 521 → 631
Test: 632 → 743


In [94]:
for file in [TRAIN_FILE, VALIDATION_FILE, TEST_FILE]:
    if os.path.exists(file):
        os.remove(file)

print("Old train/validation/test files removed.")

Old train/validation/test files removed.


In [96]:
chunksize = 100_000

train_rows = 0
validation_rows = 0
test_rows = 0

first_train = True
first_validation = True
first_test = True

for chunk_number, chunk in enumerate(
    pd.read_csv(PROCESSED_FILE, chunksize=chunksize),
    start=1
):

    train_chunk = chunk[
        chunk["step"] <= TRAIN_END_STEP
    ]

    validation_chunk = chunk[
        (chunk["step"] > TRAIN_END_STEP) &
        (chunk["step"] <= VALIDATION_END_STEP)
    ]

    test_chunk = chunk[
        chunk["step"] > VALIDATION_END_STEP
    ]

    # -----------------------------
    # TRAIN
    # -----------------------------

    if len(train_chunk) > 0:

        train_chunk.to_csv(
            TRAIN_FILE,
            mode="w" if first_train else "a",
            header=first_train,
            index=False
        )

        first_train = False
        train_rows += len(train_chunk)

    # -----------------------------
    # VALIDATION
    # -----------------------------

    if len(validation_chunk) > 0:

        validation_chunk.to_csv(
            VALIDATION_FILE,
            mode="w" if first_validation else "a",
            header=first_validation,
            index=False
        )

        first_validation = False
        validation_rows += len(validation_chunk)

    # -----------------------------
    # TEST
    # -----------------------------

    if len(test_chunk) > 0:

        test_chunk.to_csv(
            TEST_FILE,
            mode="w" if first_test else "a",
            header=first_test,
            index=False
        )

        first_test = False
        test_rows += len(test_chunk)

    print(
        f"Chunk {chunk_number}/64 processed"
    )

print("\n==============================")
print("Dataset splitting completed!")
print("==============================")

print("Train rows:", train_rows)
print("Validation rows:", validation_rows)
print("Test rows:", test_rows)

print(
    "Total rows:",
    train_rows + validation_rows + test_rows
)

Chunk 1/64 processed
Chunk 2/64 processed
Chunk 3/64 processed
Chunk 4/64 processed
Chunk 5/64 processed
Chunk 6/64 processed
Chunk 7/64 processed
Chunk 8/64 processed
Chunk 9/64 processed
Chunk 10/64 processed
Chunk 11/64 processed
Chunk 12/64 processed
Chunk 13/64 processed
Chunk 14/64 processed
Chunk 15/64 processed
Chunk 16/64 processed
Chunk 17/64 processed
Chunk 18/64 processed
Chunk 19/64 processed
Chunk 20/64 processed
Chunk 21/64 processed
Chunk 22/64 processed
Chunk 23/64 processed
Chunk 24/64 processed
Chunk 25/64 processed
Chunk 26/64 processed
Chunk 27/64 processed
Chunk 28/64 processed
Chunk 29/64 processed
Chunk 30/64 processed
Chunk 31/64 processed
Chunk 32/64 processed
Chunk 33/64 processed
Chunk 34/64 processed
Chunk 35/64 processed
Chunk 36/64 processed
Chunk 37/64 processed
Chunk 38/64 processed
Chunk 39/64 processed
Chunk 40/64 processed
Chunk 41/64 processed
Chunk 42/64 processed
Chunk 43/64 processed
Chunk 44/64 processed
Chunk 45/64 processed
Chunk 46/64 process

In [97]:
train_check = pd.read_csv(
    TRAIN_FILE,
    nrows=5
)

print("Number of columns:", train_check.shape[1])

print("\nColumns:")
for column in train_check.columns:
    print(column)

Number of columns: 25

Columns:
step
type
amount
nameOrig
oldbalanceOrg
newbalanceOrig
nameDest
oldbalanceDest
newbalanceDest
isFraud
isFlaggedFraud
user_transaction_count_before
previous_transaction_amount
time_since_previous_transaction
previous_average_amount
amount_deviation
amount_to_previous_average
balance_depletion
amount_to_balance_ratio
receiver_transaction_count_before
receiver_previous_amount
receiver_transaction_frequency
user_transfer_count_before
user_cashout_count_before
transaction_velocity


In [98]:
print(
    "Train:",
    train_check["step"].min(),
    "→",
    train_check["step"].max()
)

Train: 1 → 1


In [99]:
for name, file in [
    ("Train", TRAIN_FILE),
    ("Validation", VALIDATION_FILE),
    ("Test", TEST_FILE)
]:

    min_step = None
    max_step = None

    for chunk in pd.read_csv(
        file,
        usecols=["step"],
        chunksize=100_000
    ):

        current_min = chunk["step"].min()
        current_max = chunk["step"].max()

        min_step = (
            current_min
            if min_step is None
            else min(min_step, current_min)
        )

        max_step = (
            current_max
            if max_step is None
            else max(max_step, current_max)
        )

    print(f"{name}: {min_step} → {max_step}")

Train: 1 → 520
Validation: 521 → 631
Test: 632 → 743


In [1]:
import os
import pandas as pd

PROCESSED_FILE = "../data/processed/feature_engineered.csv"

TRAIN_FILE = "../data/processed/train_data.csv"
VALIDATION_FILE = "../data/processed/validation_data.csv"
TEST_FILE = "../data/processed/test_data.csv"

TRAIN_END_STEP = 520
VALIDATION_END_STEP = 631

# Remove old split files
for file in [TRAIN_FILE, VALIDATION_FILE, TEST_FILE]:
    if os.path.exists(file):
        os.remove(file)

print("Old split files removed.")

Old split files removed.


In [2]:
chunksize = 100_000

train_rows = 0
validation_rows = 0
test_rows = 0

first_train = True
first_validation = True
first_test = True

for chunk_number, chunk in enumerate(
    pd.read_csv(PROCESSED_FILE, chunksize=chunksize),
    start=1
):

    train_chunk = chunk[
        chunk["step"] <= TRAIN_END_STEP
    ]

    validation_chunk = chunk[
        (chunk["step"] > TRAIN_END_STEP) &
        (chunk["step"] <= VALIDATION_END_STEP)
    ]

    test_chunk = chunk[
        chunk["step"] > VALIDATION_END_STEP
    ]

    if len(train_chunk) > 0:
        train_chunk.to_csv(
            TRAIN_FILE,
            mode="w" if first_train else "a",
            header=first_train,
            index=False
        )
        first_train = False
        train_rows += len(train_chunk)

    if len(validation_chunk) > 0:
        validation_chunk.to_csv(
            VALIDATION_FILE,
            mode="w" if first_validation else "a",
            header=first_validation,
            index=False
        )
        first_validation = False
        validation_rows += len(validation_chunk)

    if len(test_chunk) > 0:
        test_chunk.to_csv(
            TEST_FILE,
            mode="w" if first_test else "a",
            header=first_test,
            index=False
        )
        first_test = False
        test_rows += len(test_chunk)

    print(f"Chunk {chunk_number}/64 processed")

print("\nDataset splitting completed!")
print("Train rows:", train_rows)
print("Validation rows:", validation_rows)
print("Test rows:", test_rows)
print(
    "Total rows:",
    train_rows + validation_rows + test_rows
)

Chunk 1/64 processed
Chunk 2/64 processed
Chunk 3/64 processed
Chunk 4/64 processed
Chunk 5/64 processed
Chunk 6/64 processed
Chunk 7/64 processed
Chunk 8/64 processed
Chunk 9/64 processed
Chunk 10/64 processed
Chunk 11/64 processed
Chunk 12/64 processed
Chunk 13/64 processed
Chunk 14/64 processed
Chunk 15/64 processed
Chunk 16/64 processed
Chunk 17/64 processed
Chunk 18/64 processed
Chunk 19/64 processed
Chunk 20/64 processed
Chunk 21/64 processed
Chunk 22/64 processed
Chunk 23/64 processed
Chunk 24/64 processed
Chunk 25/64 processed
Chunk 26/64 processed
Chunk 27/64 processed
Chunk 28/64 processed
Chunk 29/64 processed
Chunk 30/64 processed
Chunk 31/64 processed
Chunk 32/64 processed
Chunk 33/64 processed
Chunk 34/64 processed
Chunk 35/64 processed
Chunk 36/64 processed
Chunk 37/64 processed
Chunk 38/64 processed
Chunk 39/64 processed
Chunk 40/64 processed
Chunk 41/64 processed
Chunk 42/64 processed
Chunk 43/64 processed
Chunk 44/64 processed
Chunk 45/64 processed
Chunk 46/64 process

In [3]:
train_check = pd.read_csv(
    TRAIN_FILE,
    nrows=5
)

print("Number of columns:", train_check.shape[1])

print("\nNew behavioural features:")

new_features = [
    "receiver_transaction_count_before",
    "receiver_previous_amount",
    "receiver_transaction_frequency",
    "user_transfer_count_before",
    "user_cashout_count_before",
    "transaction_velocity"
]

print(train_check[new_features])


Number of columns: 25

New behavioural features:
   receiver_transaction_count_before  receiver_previous_amount  \
0                                  0                       NaN   
1                                  0                       NaN   
2                                  0                       NaN   
3                                  0                       NaN   
4                                  0                       NaN   

   receiver_transaction_frequency  user_transfer_count_before  \
0                             NaN                           0   
1                             NaN                           0   
2                             NaN                           0   
3                             NaN                           0   
4                             NaN                           0   

   user_cashout_count_before  transaction_velocity  
0                          0                     0  
1                          0                     1  
2   